# OpenFold 3 Structure Prediction on Google Cloud with `foldrun-shim`

<a href="https://colab.research.google.com/github/aqlaboratory/openfold/blob/main/notebooks/OpenFold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Upstream OpenFold Colab"/></a>

This notebook adapts the official [**AlQuraishi Lab OpenFold 3 (`aqlaboratory/openfold-3`)**](https://github.com/aqlaboratory/openfold-3) inference workflows ([`examples/example_inference_inputs/`](https://github.com/aqlaboratory/openfold-3/tree/main/examples/example_inference_inputs), [`examples/example_runner_yamls/`](https://github.com/aqlaboratory/openfold-3/tree/main/examples/example_runner_yamls), and the [OpenFold 3 Documentation](https://openfold-3.readthedocs.io/en/latest/inference.html)) and the public [OpenFold Colab Notebook](https://github.com/aqlaboratory/openfold/blob/main/notebooks/OpenFold.ipynb) to run end-to-end on **FoldRun** via **`foldrun-shim`**.

With `foldrun-shim`, both unmodified CLI invocations (`!run_openfold predict ...`) and Python notebook calls (`from foldrun_shim import openfold3; openfold3.predict(...)`) execute on FoldRun's **auto-scaling Vertex AI A100 Pipelines** backed by a pre-provisioned **2.5 TB SSD Cloud Filestore NFS share (`/mnt/nfs/foldrun`)** containing all **9 OpenFold 3 reference databases**.


---

## Original Notebook Reference, Credits & Citations

- **Original Public Notebook & Examples**:
  - Official OpenFold Colab Notebook: [`aqlaboratory/openfold` `notebooks/OpenFold.ipynb`](https://github.com/aqlaboratory/openfold/blob/main/notebooks/OpenFold.ipynb) ([Open Original in Colab](https://colab.research.google.com/github/aqlaboratory/openfold/blob/main/notebooks/OpenFold.ipynb))
  - Official OpenFold 3 Inference Walkthrough & Examples: [`aqlaboratory/openfold-3/examples/example_inference_inputs/`](https://github.com/aqlaboratory/openfold-3/tree/main/examples/example_inference_inputs) and [`docs/source/inference_Walkthrough.md`](https://github.com/aqlaboratory/openfold-3/blob/main/docs/source/inference_Walkthrough.md)
- **Upstream Repository**: [`aqlaboratory/openfold-3`](https://github.com/aqlaboratory/openfold-3)
- **Authors & Credits**: Gustaf Ahdritz, Nazim Bouatta, Christina Floristean, Sachin Kadyan, Qisheng Xia, William Gerecke, Timothy J. O'Donnell, Daniel Berenberg, Ian Fisk, Niccolò Zanichelli, Bo Zhang, Arkadiusz Nowaczynski, Bei Wang, Marta M. Stepniewska-Dziubinska, Shang Zhang, Adegoke Ojewole, Murat Efe Guney, Stella Biderman, Andrew M. Watkins, Stephen Ra, Pablo Ribalta Lorenzo, Lucas Nivon, Brian Weitzner, Yih-En Andrew Ban, Mohammed AlQuraishi, and the **OpenFold Consortium** (**AlQuraishi Lab, Columbia University**).
- **License**: Apache License 2.0.
- **Primary Citations**:
  1. AlQuraishi Lab & OpenFold Consortium. "OpenFold3: An Open-Source All-Atom Biomolecular Structure Prediction Model." (2025). [github.com/aqlaboratory/openfold-3](https://github.com/aqlaboratory/openfold-3)
  2. Ahdritz, G. *et al.* "OpenFold: retraining AlphaFold2 yields new insights into its learning mechanisms and capacity for generalization." *Nature Methods* 21, 1514–1524 (2024). [doi:10.1038/s41592-024-02272-z](https://doi.org/10.1038/s41592-024-02272-z)

---

## Upstream OpenFold 3 vs. FoldRun Cloud Shim (`foldrun-shim`)

| Capability | Upstream Public OpenFold 3 (`aqlaboratory/openfold-3`) | FoldRun Cloud Execution (`foldrun-shim` + `foldrun-gateway`) |
| :--- | :--- | :--- |
| **Local Hardware & Drivers** | Requires local NVIDIA GPU, CUDA, `cuequivariance` / `deepspeed` kernels, `hmmer` (`jackhmmer`, `nhmmer`), and `kalign2`. | **Zero local GPUs or compiled binaries**: Runs from any CPU laptop, Colab Enterprise, or Vertex AI Workbench with only `roles/run.invoker`. |
| **Model Weights & CCD** | Requires `setup_openfold` to download `of3-p2-155k.pt` and `components.bcif` into `~/.openfold3/`. | **Pre-mounted on SSD Filestore NFS**: `/mnt/nfs/foldrun/of3/params/of3-p2-155k.pt` and `/mnt/nfs/foldrun/of3/ccd/components.bcif`. |
| **Protein + RNA MSA Generation** | Default `--use-msa-server` sends **protein sequences only** to the public ColabFold server (no RNA MSA support). Local RNA + protein MSAs require downloading TB-scale databases. | **Full Protein + RNA MSA in private VPC**: Runs `jackhmmer` against `uniref90`, `mgnify`, `pdb_seqres` (proteins) **and** `nhmmer` against `rfam`, `rnacentral` (RNA) on `c2-standard-16` VMs, cached by sequence SHA-256 in `/mnt/nfs/foldrun/of3_msas_cache/`. |
| **PDB Structural Templates** | Downloads template `.cif` structures from RCSB at runtime or requires a local mmCIF mirror in `runner.yaml`. | Automatically aligns against `pdb_seqres.txt` (`pdb_seqres.sto`) and injects `template_preprocessor_settings.structure_directory = /mnt/nfs/foldrun/pdb_mmcif/mmcif_files` (`fetch_missing_structures: false`). |
| **Multi-Seed GPU Scaling** | Runs seeds sequentially on a single local GPU unless manually orchestrated. | Fans out `--num_model_seeds` in parallel across dedicated **NVIDIA A100 40GB / 80GB GPUs** via Vertex AI `dsl.ParallelFor` with Dynamic Workload Scheduler (`FLEX_START`). |
| **Notebook Interrupt Recovery** | Kernel disconnect or browser refresh loses in-flight GPU progress. | Persists `<output_dir>/.foldrun_job.json` on `202 Accepted`; re-running a cell re-attaches to the active Vertex AI `PipelineJob` via `input_sha256`. |

## 1. Prerequisites, Authentication & Installing `foldrun-shim` from GCS

During FoldRun deployment (`./deploy-all.sh` or `./deploy-all.sh --steps build --build-target sdk`), Cloud Build packages `foldrun-shim` into a pure-Python wheel and uploads both the wheel and all 7 modified public notebooks to your project's FoldRun GCS bucket under `gs://<PROJECT_ID>-foldrun-data/sdk/`:
- **Pre-Built Wheel**: `gs://<PROJECT_ID>-foldrun-data/sdk/latest_build` (and dated incremental builds)
- **Modified Notebooks**: `gs://<PROJECT_ID>-foldrun-data/sdk/notebooks/<model>/*.ipynb`

### Step-by-Step Setup (Executed in the Next Cell)

1. **Step 1 — Authenticate to Google Cloud First**:
   - **Google Colab**: Calls `from google.colab import auth; auth.authenticate_user()` so `gcloud storage cp` and `gcloud auth print-identity-token` have valid credentials before downloading the wheel.
   - **Colab Enterprise / Vertex AI Workbench**: Authenticated automatically via the runtime's attached service account and the Google Cloud Metadata Server.
   - **Local Jupyter Kernel**: Authenticate beforehand with `gcloud auth login`.
2. **Step 2 — Automatic Configuration Loading (`.env`)**:
   - FoldRun's deployment script (`./deploy-all.sh`) generates local `.env` files in `sdk/foldrun-shim/notebooks/` and uploads `gs://${FOLDRUN_GCS_BUCKET}/sdk/.env`.
   - The cell below automatically loads `.env` (searching `./.env`, `../.env`, `/content/.env`, or downloading from `gs://${FOLDRUN_GCS_BUCKET}/sdk/.env`), populating `GCP_PROJECT_ID`, `GCP_REGION`, `FOLDRUN_GCS_BUCKET`, `FOLDRUN_GATEWAY_URL`, and `FOLDRUN_VIEWER_URL`.
3. **Step 3 — Download & Install `foldrun-shim` from the FoldRun GCS Bucket**:
   Requires `roles/storage.objectViewer` on `gs://<PROJECT_ID>-foldrun-data` (ideally scoped to the `sdk/` prefix via an IAM condition):
   ```bash
   gcloud storage cp gs://${FOLDRUN_GCS_BUCKET}/sdk/latest_build /tmp/
   WHL=$(cat /tmp/latest_build)
   gcloud storage cp gs://${FOLDRUN_GCS_BUCKET}/sdk/${WHL} /tmp/
   pip install -q /tmp/${WHL}
   ```
4. **Step 4 — Configure `FOLDRUN_GATEWAY_URL` & OIDC Gateway Auth (`roles/run.invoker`)**:
   `foldrun_shim.auth.fetch_gateway_headers()` automatically attaches `Authorization: Bearer <OIDC_ID_TOKEN>` to every gateway request via:
   1. `FOLDRUN_ID_TOKEN` (if explicitly set),
   2. `FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT` (`gcloud auth print-identity-token --impersonate-service-account=...`),
   3. Google Cloud Metadata Server (`google.oauth2.id_token.fetch_id_token` on Vertex AI Workbench / Colab Enterprise),
   4. `gcloud auth print-identity-token` fallback (Colab / local session).

In [ ]:
# @title 1.1 Authenticate to Google Cloud & Install `foldrun-shim` from GCS
import json
import os
import re
import shutil
import subprocess
import sys
import sysconfig
from pathlib import Path


def _ensure_scripts_on_path() -> None:
    """Ensure pip console scripts (including --user installs) are discoverable on PATH."""
    script_dirs: list[str] = []
    for scheme in (None, f"{os.name}_user"):
        try:
            p = (
                sysconfig.get_path("scripts")
                if scheme is None
                else sysconfig.get_path("scripts", scheme)
            )
            if p:
                script_dirs.append(p)
        except Exception:
            pass
    script_dirs.append(str(Path(sys.executable).parent))
    current_parts = os.environ.get("PATH", "").split(os.pathsep) if os.environ.get("PATH") else []
    for d in reversed(script_dirs):
        if d and d not in current_parts:
            current_parts.insert(0, d)
    os.environ["PATH"] = os.pathsep.join(current_parts)


def _is_placeholder_value(val: str) -> bool:
    lowered = (val or "").strip().lower()
    return not lowered or any(
        tok in lowered
        for tok in (
            "your-project-foldrun-data",
            "your-project",
            "your_project",
            "example",
            "<",
        )
    )


def _is_placeholder_bucket(bucket: str) -> bool:
    clean = (bucket or "").strip().removeprefix("gs://").rstrip("/")
    return _is_placeholder_value(clean)


_ensure_scripts_on_path()


def _load_foldrun_dotenv(custom_path: Path | None = None) -> Path | None:
    """Load key=value pairs from a local .env file generated by deploy-all.sh."""
    if custom_path is not None:
        candidates = [custom_path]
    else:
        candidates = []
        if os.environ.get("FOLDRUN_ENV_FILE"):
            candidates.append(Path(os.environ["FOLDRUN_ENV_FILE"]))
        candidates.extend(
            [Path(".env"), Path("../.env"), Path("../../.env"), Path("/content/.env")]
        )
    loaded_from: Path | None = None
    for path in candidates:
        if not path.is_file():
            continue
        for raw_line in path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            k = re.sub(r"^export(?=\s)\s*", "", k.strip())
            v = v.strip().strip("'").strip('"')
            if k and v:
                existing = os.environ.get(k, "")
                if not existing or _is_placeholder_value(existing):
                    os.environ[k] = v
        if loaded_from is None:
            loaded_from = path.resolve()
        if custom_path is not None or not _is_placeholder_value(
            os.environ.get("FOLDRUN_GATEWAY_URL", "")
        ):
            return path.resolve()
    return loaded_from


_dotenv_path = _load_foldrun_dotenv()
if _dotenv_path:
    print(f"Loaded FoldRun configuration from {_dotenv_path}")

os.environ.setdefault("FOLDRUN_GCS_BUCKET", "your-project-foldrun-data")
os.environ.setdefault("FOLDRUN_GATEWAY_URL", "https://foldrun-gateway-example-uc.a.run.app")
# Optional: Impersonate a service account granted roles/run.invoker on foldrun-gateway
# os.environ["FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT"] = "foldrun-notebook-invoker@my-project.iam.gserviceaccount.com"

# 1. Authenticate to Google Cloud first (required in standard Google Colab before accessing GCS)
if "google.colab" in sys.modules:
    from google.colab import auth as colab_auth

    colab_auth.authenticate_user()


def _install_foldrun_shim_from_gcs(gcs_bucket: str) -> None:
    """Download latest foldrun_shim wheel via latest_build from gs://<bucket>/sdk/ and install via pip."""
    clean_bucket = (gcs_bucket or "").strip().removeprefix("gs://").rstrip("/")
    bucket_configured = not _is_placeholder_bucket(clean_bucket)

    if bucket_configured and _is_placeholder_value(os.environ.get("FOLDRUN_GATEWAY_URL", "")):
        try:
            env_dst = Path("/tmp/foldrun_sdk.env")
            res = subprocess.run(
                [
                    "gcloud",
                    "storage",
                    "cp",
                    f"gs://{clean_bucket}/sdk/.env",
                    str(env_dst),
                ],
                check=False,
                capture_output=True,
                text=True,
            )
            if res.returncode == 0 and env_dst.is_file():
                if _load_foldrun_dotenv(env_dst):
                    print(f"Loaded remote FoldRun configuration from gs://{clean_bucket}/sdk/.env")
        except Exception as exc:
            print(f"Notice: could not load gs://{clean_bucket}/sdk/.env: {exc}")

    whl_filename = "foldrun_shim-0.1.0-py3-none-any.whl"
    if bucket_configured:
        try:
            latest_build_dst = Path("/tmp/foldrun_latest_build")
            res = subprocess.run(
                [
                    "gcloud",
                    "storage",
                    "cp",
                    f"gs://{clean_bucket}/sdk/latest_build",
                    str(latest_build_dst),
                ],
                check=False,
                capture_output=True,
                text=True,
            )
            if res.returncode == 0 and latest_build_dst.is_file():
                candidate = latest_build_dst.read_text(encoding="utf-8").strip()
                if candidate.endswith(".whl"):
                    whl_filename = candidate
                    print(
                        f"Detected latest FoldRun SDK build from gs://{clean_bucket}/sdk/latest_build: {whl_filename}"
                    )
        except Exception as exc:
            print(f"Notice: could not read gs://{clean_bucket}/sdk/latest_build: {exc}")

    whl_path = Path(f"/tmp/{whl_filename}")
    marker_path = Path(f"/tmp/{whl_filename}.installed")
    if marker_path.exists():
        try:
            import foldrun_shim

            _ensure_scripts_on_path()
            return
        except ImportError:
            pass

    if not bucket_configured:
        try:
            import foldrun_shim

            _ensure_scripts_on_path()
            print(
                "Notice: FOLDRUN_GCS_BUCKET is unconfigured; using locally installed foldrun_shim."
            )
            return
        except ImportError:
            raise RuntimeError(
                f"foldrun_shim is not installed. Set FOLDRUN_GCS_BUCKET='<PROJECT_ID>-foldrun-data' "
                f"so this cell can download gs://<PROJECT_ID>-foldrun-data/sdk/{whl_filename}."
            )

    gcs_uri = f"gs://{clean_bucket}/sdk/{whl_filename}"
    print(f"Downloading {whl_filename} from {gcs_uri}...")
    cp_res = subprocess.run(
        ["gcloud", "storage", "cp", gcs_uri, str(whl_path)],
        check=False,
        capture_output=True,
        text=True,
    )
    if cp_res.returncode != 0:
        err_line = (
            (cp_res.stderr or cp_res.stdout or "gcloud storage cp failed").strip().splitlines()[-1]
        )
        try:
            import foldrun_shim  # noqa: F401

            _ensure_scripts_on_path()
            print(
                f"Warning: could not download {gcs_uri} ({err_line}); using locally installed foldrun_shim."
            )
            return
        except ImportError:
            raise RuntimeError(
                f"Failed to download {gcs_uri} ({err_line}) and foldrun_shim is not installed locally."
            )

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", str(whl_path)],
        check=True,
    )
    _ensure_scripts_on_path()
    marker_path.write_text("installed", encoding="utf-8")
    print(f"Installed {whl_filename} successfully.")


_install_foldrun_shim_from_gcs(os.environ["FOLDRUN_GCS_BUCKET"])
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "py3Dmol",
        "matplotlib",
        "numpy",
        "pyyaml",
    ],
    check=False,
)

import yaml

from foldrun_shim.auth import resolve_gateway_url

WORK_DIR = Path("./of3_workspace").resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Resolved FoldRun Gateway URL : {resolve_gateway_url()}")
print(
    f"run_openfold CLI in PATH     : {shutil.which('run_openfold') or 'available via foldrun_shim.cli'}"
)
print(f"Notebook Working Directory   : {WORK_DIR}")

In [ ]:
# @title 1B. (Optional) Check Live GPU Availability & Configure On-Demand vs. DWS Scheduling
# @markdown Thin-permission notebook users (who only have `roles/run.invoker` on the FoldRun Gateway)
# @markdown can inspect live GPU availability, active/queued jobs, and regional quotas directly from Python
# @markdown without needing GCP project-level GCE or ServiceUsage IAM permissions.
# @markdown
# @markdown **CLI Equivalents:**
# @markdown - `!foldrun gpus` (inspect live GPU availability and queue depths)
# @markdown - `!foldrun status <job_id>` (inspect status of an active or queued job)
# @markdown - `!foldrun cancel <job_id>` (cancel a queued or running job)
# @markdown - GCP Project Admins only: `!gcloud alpha services quota list --service=aiplatform.googleapis.com --consumer=projects/$GCP_PROJECT_ID`

import os

try:
    from IPython.display import display
except ImportError:
    display = print

# 1. Query live GPU availability and queue status via FoldRun Gateway
try:
    import foldrun_shim

    gpu_status = foldrun_shim.check_gpus()
    display(gpu_status)
except Exception as exc:
    print(f"Notice: Could not query live GPU status via FoldRun Gateway: {exc}")
    print(
        "You can still proceed with prediction using default scheduling or environment variables below."
    )

# 2. Scheduling & Hardware Configuration (Optional Overrides)
# Uncomment or adjust any of the following environment variables to control hardware & scheduling:

# GPU Architecture: OpenFold 3 requires "A100" (40GB, default via "auto") or "A100_80GB"
# os.environ["FOLDRUN_GPU_TYPE"] = "A100"

# Scheduling Strategy:
# - "true"  -> Dynamic Workload Scheduler (DWS) FLEX_START (saves 45-68% cost; may queue if regional DWS capacity is constrained)
# - "false" -> Immediate On-Demand STANDARD GPUs (provisions immediately without DWS queueing)
# os.environ["FOLDRUN_ENABLE_FLEX_START"] = "false"

# Optional Queue Timeout: Automatically cancel the job if queued in DWS longer than N seconds
# os.environ["FOLDRUN_QUEUE_TIMEOUT_SEC"] = "300"

---
## Part A — Original Upstream Example (Unmodified / Verbatim Workflow)

### Original `aqlaboratory/openfold-3` Upstream Examples

In the official [`aqlaboratory/openfold-3` repository](https://github.com/aqlaboratory/openfold-3), inference inputs are defined in `InferenceQuerySet` JSON format (`examples/example_inference_inputs/query_ubiquitin.json` and `query_protein_ligand.json`).

Below we write out the upstream queries and execute the exact upstream command:
```bash
!run_openfold predict --query_json=query_ubiquitin.json --runner_yaml=runner.yaml --output_dir=./of3_ubiquitin_out
```


In [ ]:
# Write exact upstream aqlaboratory/openfold-3 example query_ubiquitin.json and query_protein_ligand.json
from pathlib import Path

ubiquitin_query = {
    "queries": {
        "ubiquitin": {
            "chains": [
                {
                    "molecule_type": "protein",
                    "chain_ids": ["A"],
                    "sequence": "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG",
                }
            ]
        }
    }
}
Path("query_ubiquitin.json").write_text(json.dumps(ubiquitin_query, indent=2), encoding="utf-8")

# Upstream protein-ligand complex (examples/example_inference_inputs/query_protein_ligand.json)
protein_ligand_query = {
    "queries": {
        "protein_ligand": {
            "chains": [
                {
                    "molecule_type": "protein",
                    "chain_ids": ["A"],
                    "sequence": "GDDELYRQSLEIISRYLREQATGAKDTKPMGRSGATSRKALETLRRVGDGVQRNHETAFQGMLRKLDIKNEDDVKSLSRVMIHVFSDGVTNWGRIVTLISFGAFVAKHLKTINQESCIEPLAESITDVLVRTKRDWLVKQRGWDGFVEFFHVEDLEGG",
                },
                {
                    "molecule_type": "ligand",
                    "chain_ids": ["B"],
                    "smiles": "CC(=O)OC1C[NH+]2CCC1CC2",
                },
            ]
        }
    }
}
Path("query_protein_ligand.json").write_text(
    json.dumps(protein_ligand_query, indent=2), encoding="utf-8"
)

# Minimal runner.yaml enabling PAE head and pLDDT/PAE output
Path("runner.yaml").write_text(
    "model_update:\n  presets:\n    - predict\n    - pae_enabled\nplddt: true\npae: true\n",
    encoding="utf-8",
)
print("Wrote query_ubiquitin.json, query_protein_ligand.json, and runner.yaml")

In [ ]:
# Verbatim upstream OpenFold 3 command execution:
!run_openfold predict --query_json=query_ubiquitin.json --runner_yaml=runner.yaml --num_diffusion_samples=5 --output_dir=./of3_ubiquitin_out

---
## Part B — FoldRun Additions (`foldrun-shim` Python API, `/mnt/nfs/foldrun` Shared Databases & Weights, Local Asset Bundling, and Multi-GPU Seed Parallelism)

FoldRun expands OpenFold 3 into an institutional multi-modal platform:
1. **Full 9-Database Filestore NFS Complex (`/mnt/nfs/foldrun`)**: Covers `uniref90`, `mgnify`, `pdb_seqres`, `rfam`, `rnacentral`, `pdb_mmcif`, and `of3/ccd/components.bcif` for protein, RNA (`nhmmer`), DNA, and ligand co-folding.
2. **Local Pre-computed MSA (`.a3m`) & Stockholm (`.sto`) Bundling**: Bundles custom alignments and rewrites query JSON paths to GCS FUSE paths.
3. **Multi-Seed GPU Fan-Out**: Predicts across multiple seeds (`--num_model_seeds=5`) in parallel on Vertex AI A100 GPUs.
4. **Python Notebook Helper API (`openfold3.predict`)**: Clean programmatic interface with status cards and automatic output extraction.


## 2. How FoldRun Uses the 9 Databases on Cloud Filestore NFS (`/mnt/nfs/foldrun`)

OpenFold 3 is the most database-intensive biomolecular model in FoldRun. While AlphaFold 2 folds only amino acid sequences and Boltz-2 relies on protein MSAs + CCD, OpenFold 3 natively models **proteins, RNA, DNA, and small-molecule ligands (`ccd_codes` and `smiles`)**.

Under the hood, FoldRun pre-stages **9 reference databases** onto a 2.5 TB `BASIC_SSD` Cloud Filestore NFS share mounted at `/mnt/nfs/foldrun` across all Vertex AI worker VMs (configured in [`foldrun-agent/foldrun_app/databases.yaml`](../../../foldrun-agent/foldrun_app/databases.yaml)):

| # | Database Key (`databases.yaml`) | Filestore NFS Path (`/mnt/nfs/foldrun/...`) | Pipeline Stage | How OpenFold 3 Uses It in FoldRun |
| :-: | :--- | :--- | :--- | :--- |
| **1** | `of3_params` | `of3/params/of3-p2-155k.pt` | `predict_of3` (A100 GPU) | Official OpenFold 3 model checkpoint passed via `--inference_ckpt_path=/mnt/nfs/foldrun/of3/params/of3-p2-155k.pt`. |
| **2** | `of3_ccd` | `of3/ccd/components.bcif` | `predict_of3` (A100 GPU) | Binary CIF Chemical Component Dictionary for ligand chemical graphs (`ccd_codes`, `smiles`) and non-canonical residues. |
| **3** | `uniref90` | `uniref90/uniref90.fasta` | `msa_pipeline_of3` (`c2-standard-16`) | Searched via `jackhmmer --noali -N 1 --cpu 8` for every `molecule_type: "protein"` chain $\rightarrow$ `uniref90_hits.sto`. |
| **4** | `mgnify` | `mgnify/mgy_clusters_2022_05.fa` | `msa_pipeline_of3` (`c2-standard-16`) | Searched via `jackhmmer --noali -N 1 --cpu 8` for every `molecule_type: "protein"` chain $\rightarrow$ `mgnify_hits.sto`. |
| **5** | `pdb_seqres` | `pdb_seqres/pdb_seqres.txt` | `msa_pipeline_of3` (`c2-standard-16`) | Searched via `jackhmmer -N 1 --cpu 8` when `--use_templates=true` $\rightarrow$ `pdb_seqres.sto` (consumed by OpenFold 3's `StoParser` as `template_alignment_file_path`). |
| **6** | `uniprot` | `uniprot/uniprot.fasta` | `msa_pipeline_of3` (`c2-standard-16`) | Merged UniProtKB (TrEMBL + Swiss-Prot) mounted in `ref_databases` metadata for cross-chain species pairing. |
| **7** | `pdb_mmcif` | `pdb_mmcif/mmcif_files/*.cif` | `predict_of3` (A100 GPU) | 220,000+ experimental PDB structures in flat `.cif` format; wired into `runner.yaml` as `template_preprocessor_settings.structure_directory`. |
| **8** | `rfam` | `rfam/rfam.fasta` | `msa_pipeline_of3` (`c2-standard-16`) | Searched via `nhmmer --noali --cpu 8` for every `molecule_type: "rna"` chain $\rightarrow$ `rfam_hits.sto`. |
| **9** | `rnacentral` | `rnacentral/rnacentral.fasta` | `msa_pipeline_of3` (`c2-standard-16`) | Searched via `nhmmer --noali --cpu 8` for every `molecule_type: "rna"` chain $\rightarrow$ `rnacentral_hits.sto`. |

### Content-Addressed Per-Chain MSA Caching (`/mnt/nfs/foldrun/of3_msas_cache/`)

In [`msa_pipeline.py`](../../../foldrun-agent/foldrun_app/models/of3/pipeline/components/msa_pipeline.py), FoldRun hashes each chain's sequence (`seq_hash = hashlib.sha256(sequence.encode()).hexdigest()`) and checks `/mnt/nfs/foldrun/of3_msas_cache/{molecule_type}_{seq_hash}/`:
- **Protein cache directory (`protein_<sha256>/`)**: Stores `uniref90_hits.sto`, `mgnify_hits.sto`, and `pdb_seqres.sto` (when `use_templates=True`). Note the `_hits.sto` suffix, which satisfies OpenFold 3's `MSASettings.max_seq_counts` basename filter.
- **RNA cache directory (`rna_<sha256>/`)**: Stores `rfam_hits.sto` and `rnacentral_hits.sto` generated by `nhmmer`.
- Whenever a protein or RNA chain is reused across different complexes (e.g., screening a target protein or ribonucleoprotein against dozens of small-molecule ligands), `msa_pipeline_of3` detects a cache hit on `/mnt/nfs/foldrun/of3_msas_cache/` and injects the cached `.sto` paths into `updated_query_json` in milliseconds.

In [ ]:
# @title 2.1 Author OpenFold 3 `query.json` Inputs (Adapted from `aqlaboratory/openfold-3`)
# 1. Upstream Example A: Monomer Protein (Ubiquitin, adapted from examples/example_inference_inputs/query_ubiquitin.json)
ubiquitin_query = {
    "queries": {
        "ubiquitin": {
            "chains": [
                {
                    "molecule_type": "protein",
                    "chain_ids": ["A"],
                    "sequence": "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG",
                }
            ]
        }
    }
}
ubiquitin_json_path = WORK_DIR / "query_ubiquitin.json"
ubiquitin_json_path.write_text(json.dumps(ubiquitin_query, indent=2), encoding="utf-8")

# 2. Upstream Example B: Multi-Modal Protein + RNA + DNA + Ligand Complex (query.json)
# Demonstrates all 9 FoldRun NFS databases in a single OpenFold 3 InferenceQuerySet:
#   - Chain A (protein): Jackhmmer against uniref90, mgnify, and pdb_seqres + templates from pdb_mmcif/mmcif_files
#   - Chain B (rna):     nhmmer against rfam/rfam.fasta and rnacentral/rnacentral.fasta (not supported by public ColabFold server!)
#   - Chain C (dna):     DNA chain folded MSA-free per AlphaFold 3 / OpenFold 3 protocol
#   - Chain L (ligand):  ATP cofactor specified via PDB Chemical Component Dictionary code (ccd_codes -> of3/ccd/components.bcif)
#   - Chain M (ligand):  Small-molecule inhibitor specified via SMILES string (adapted from query_protein_ligand.json)
multimodal_query = {
    "queries": {
        "rnp_atp_complex": {
            "chains": [
                {
                    "molecule_type": "protein",
                    "chain_ids": ["A"],
                    "sequence": (
                        "GDDELYRQSLEIISRYLREQATGAKDTKPMGRSGATSRKALETLRRVGDGVQRNHETAFQ"
                        "GMLRKLDIKNEDDVKSLSRVMIHVFSDGVTNWGRIVTLISFGAFVAKHLKTINQESCIEP"
                        "LAESITDVLVRTKRDWLVKQRGWDGFVEFFHVEDLEGG"
                    ),
                },
                {
                    "molecule_type": "rna",
                    "chain_ids": ["B"],
                    "sequence": "GGGGCCACUAGGGACAGGAUGUUUUAGAGCUAGAAAUAGCAAGUUAAAAUAAGGCUAGUCCGUUAUCAACUUGAAAAAGUG",
                },
                {
                    "molecule_type": "dna",
                    "chain_ids": ["C"],
                    "sequence": "TCCTGTAGTGACATATTCACAAACGTGAAG",
                },
                {
                    "molecule_type": "ligand",
                    "chain_ids": ["L"],
                    "ccd_codes": ["ATP"],
                },
                {
                    "molecule_type": "ligand",
                    "chain_ids": ["M"],
                    "smiles": "CC(=O)OC1C[NH+]2CCC1CC2",
                },
            ]
        }
    }
}

query_json_path = Path("query.json")
query_json_path.write_text(json.dumps(multimodal_query, indent=2), encoding="utf-8")
print(f"Wrote multi-modal OpenFold 3 query to {query_json_path.resolve()}:")
print(query_json_path.read_text(encoding="utf-8"))

In [ ]:
# @title 2.2 Create a Custom `runner.yaml` with Allowlisted Scientific Overrides
# FoldRun's gateway allowlist (foldrun_app/gateway/flag_allowlist.py) permits the following
# top-level scientific keys in runner.yaml:
#   - model_update        (e.g., presets: [predict, pae_enabled] or [predict, low_mem])
#   - low_mem             (bool: sequential pairformer embedding mode for large complexes)
#   - plddt               (bool: enable pLDDT confidence head)
#   - pae                 (bool: enable Predicted Aligned Error confidence head)
#   - experiment_settings (dict: custom experiment/seed settings)
#
# Security note: Host/container filesystem path keys (structure_directory, inference_ckpt_path,
# ccd_file_path, output_dir) are blocked in user runner.yaml files. When --use_templates=true,
# FoldRun's predict_of3 component automatically deep-merges:
#   template_preprocessor_settings:
#     structure_directory: /mnt/nfs/foldrun/pdb_mmcif/mmcif_files
#     fetch_missing_structures: false
#     structure_file_format: cif

runner_yaml_config = {
    "model_update": {
        "presets": ["predict", "pae_enabled"],
    },
    "low_mem": False,
    "plddt": True,
    "pae": True,
    "experiment_settings": {
        "seeds": [42, 100],
    },
}

runner_yaml_path = Path("runner.yaml")
runner_yaml_path.write_text(
    yaml.safe_dump(runner_yaml_config, sort_keys=False),
    encoding="utf-8",
)
print(f"Wrote custom runner.yaml to {runner_yaml_path.resolve()}:")
print(runner_yaml_path.read_text(encoding="utf-8"))

## 3. Interface 1: Drop-In Native CLI (`!run_openfold predict ...`)

`foldrun-shim` registers the `run_openfold` console script entrypoint (`foldrun_shim.cli:openfold3_main`). You can execute standard OpenFold 3 shell commands directly inside Jupyter / Colab cells using `!run_openfold predict ...`.

### How `foldrun-shim` Sanitizes Local CLI Invocations (`_strip_local_openfold3_args`)
When migrating existing shell scripts or public notebooks to FoldRun:
- **Automatically Stripped Host Flags**: `--inference_ckpt_path` and `--data_dir` are stripped locally by `_strip_local_openfold3_args()` in `foldrun_shim/cli.py` because FoldRun automatically mounts `/mnt/nfs/foldrun/of3/params/of3-p2-155k.pt` and `/mnt/nfs/foldrun` inside the Vertex AI containers.
- **Client-Handled Flags**: `--query_json`, `--output_dir` (or `--out_dir`), `--runner_yaml`, and `--override` are processed by `foldrun-shim` to bundle local files and manage `<output_dir>/.foldrun_job.json`.
- **Validated Scientific Flags Forwarded to Vertex AI (`_validate_openfold3_flags`)**:
  - `--num_diffusion_samples` (`1..25`, default `5`)
  - `--num_model_seeds` (`1..10`, default `1`)
  - `--use_templates` (`true` / `false`, default `true`)
  - `--use_msa_server` (`true` / `false`, default `true`)
  - `--seed` (`int`, mapped to KFP `base_seed`)

In [ ]:
# @title 3.1 Inspect CLI Argument Sanitization (`_strip_local_openfold3_args`)
from foldrun_shim.cli import _strip_local_openfold3_args

# Demonstrate how legacy local scripts with --inference_ckpt_path and --data_dir are sanitized:
legacy_cli_tokens = [
    "predict",
    "--query_json=query.json",
    "--runner_yaml=runner.yaml",
    "--output_dir=./of3_out",
    "--inference_ckpt_path=/home/user/.openfold3/of3-p2-155k.pt",
    "--data_dir=/mnt/local_ssd/openfold_dbs",
    "--num_diffusion_samples=5",
    "--num_model_seeds=2",
    "--use_templates=true",
]
q_json, out_dir, r_yaml, override_flag, scientific_args = _strip_local_openfold3_args(
    legacy_cli_tokens
)
print("Parsed query_json      :", q_json)
print("Parsed output_dir      :", out_dir)
print("Parsed runner_yaml     :", r_yaml)
print("Forwarded CLI args     :", scientific_args)

In [ ]:
# @title 3.2 Run Drop-In Native CLI (`!run_openfold predict ...`)
!run_openfold predict --query_json=query.json --runner_yaml=runner.yaml --output_dir=./of3_out --num_diffusion_samples=5 --num_model_seeds=2 --use_templates=true

## 4. Interface 2: Python Notebook Helper API (`from foldrun_shim import openfold3`)

For interactive notebook workflows, `foldrun_shim` provides the `openfold3.predict(...)` helper (`OpenFold3NotebookHelper` in [`foldrun_shim/client.py`](../../foldrun_shim/client.py)):

```python
from foldrun_shim import openfold3

job = openfold3.predict(
    "query.json",
    output_dir="./of3_out",
    runner_yaml="runner.yaml",
    num_diffusion_samples=5,
    num_model_seeds=2,
    use_templates=True,
)
job  # Renders an interactive HTML status card in Jupyter / Colab
```

### What Happens During `openfold3.predict(...)`
1. **Asset Bundling & Deterministic Hashing**: `bundle_local_inputs()` reads `query.json` and `runner.yaml`, bundles any local `.a3m`/`.sto` files referenced in `main_msa_file_paths`, `paired_msa_file_path`, or `template_alignment_file_path` into an in-memory `assets.tar.gz`, and computes `client_input_sha256`.
2. **Interrupt-Resume Check (`.foldrun_job.json`)**: If `./of3_out/.foldrun_job.json` exists with a matching `input_sha256` and the remote Vertex AI `PipelineJob` is still `PENDING`, `RUNNING`, or `SUCCEEDED`, `foldrun-shim` re-attaches immediately (`job.deduplicated = True`) without submitting a duplicate A100 job.
3. **Parallel Multi-Seed Execution on Vertex AI**: `ConfigureSeedsOF3` generates `num_model_seeds` seeds, `MSAPipelineOF3` runs `jackhmmer` (protein) and `nhmmer` (RNA) against `/mnt/nfs/foldrun`, and `dsl.ParallelFor` launches `num_model_seeds` parallel `PredictOF3` tasks on NVIDIA A100 GPUs (auto-selecting `a2-highgpu-1g` 40GB or `a2-ultragpu-1g` 80GB based on `count_tokens(query_json)`).
4. **Streaming Output Extraction**: Upon completion, `foldrun-gateway` packages per-seed structures and confidence JSONs into `native_outputs.tar.gz`, returns a 15-minute GCS V4 Signed URL, and `foldrun-shim` extracts the native `<query_name>/seed_<seed>/...` hierarchy into `output_dir`.

In [ ]:
# @title 4.1 Run `openfold3.predict(...)` & Render `GatewayJobResult` HTML Card
try:
    from IPython.display import display
except ImportError:

    def display(obj):
        print(obj._repr_html_() if hasattr(obj, "_repr_html_") else obj)


from foldrun_shim import openfold3
from foldrun_shim.client import GatewayJobResult

output_dir = Path("./of3_out").resolve()

if "example-uc.a.run.app" not in os.environ.get("FOLDRUN_GATEWAY_URL", ""):
    # Live submission to FoldRun Gateway:
    job = openfold3.predict(
        "query.json",
        output_dir="./of3_out",
        runner_yaml="runner.yaml",
        num_diffusion_samples=5,
        num_model_seeds=2,
        use_templates=True,
    )
else:
    # Offline preview of the GatewayJobResult object and .foldrun_job.json state token:
    output_dir.mkdir(parents=True, exist_ok=True)
    demo_state = {
        "job_id": "openfold3-gw-8f4a-9c12-20261024083000",
        "model": "openfold3",
        "input_sha256": "8f4a19c0d2e3b4a567890123456789abcdef0123456789abcdef0123456789ab",
        "server_input_sha256": "8f4a19c0d2e3b4a567890123456789abcdef0123456789abcdef0123456789ab",
        "viewer_url": "https://foldrun-viewer-example-uc.a.run.app/job/openfold3-gw-8f4a-9c12-20261024083000",
        "submitted_at": "2026-10-24T08:30:00+00:00",
    }
    (output_dir / ".foldrun_job.json").write_text(
        json.dumps(demo_state, indent=2), encoding="utf-8"
    )
    job = GatewayJobResult(
        job_id=demo_state["job_id"],
        model="openfold3",
        state="PIPELINE_STATE_SUCCEEDED",
        out_dir=output_dir,
        viewer_url=demo_state["viewer_url"],
        deduplicated=False,
        extracted_files=[],
    )

print("Persisted interrupt-recovery token (.foldrun_job.json):")
print((output_dir / ".foldrun_job.json").read_text(encoding="utf-8"))
display(job)

## 5. Bundling Local Pre-Computed MSAs (`main_msa_file_paths`), Template Alignments (`template_alignment_file_path`) & Custom `runner.yaml`

In many drug discovery campaigns, you may already have a curated local alignment (`.a3m` or `.sto`) or a custom Stockholm template alignment (`.sto`) for one chain in a complex, while wanting FoldRun to search `/mnt/nfs/foldrun` (`rfam`, `rnacentral`, `uniref90`, `mgnify`) for the remaining chains.

### How `foldrun-shim` and `foldrun-gateway` Handle Local Assets + Cloud Databases Together
1. **Client-Side Asset Collection (`_collect_openfold3_assets` in `foldrun_shim/bundler.py`)**:
   - Inspects every chain in `queries.<query_name>.chains[*]`:
     - `main_msa_file_paths` *(list of relative paths to `.a3m` or `.sto` files)*
     - `paired_msa_file_path` *(optional relative path to a cross-chain paired `.a3m` or `.sto` file)*
     - `template_alignment_file_path` *(optional relative path to a Stockholm `.sto` template alignment)*
   - Verifies that every referenced file exists inside the `query.json` base directory (blocking `..` path traversal, symlinks, or absolute `/mnt/nfs/...` paths) and compresses them into `assets.tar.gz` (up to 50 files / 256 MB uncompressed).
2. **Gateway Staging & Path Rewriting (`_rewrite_openfold3_json` in `foldrun_app/gateway/staging.py`)**:
   - Extracts `assets.tar.gz` to `gs://{bucket}/gateway_jobs/{job_id}/inputs/assets/<rel_path>` and rewrites the relative paths in the staged `query.json` to Vertex AI's GCSFuse mount `/gcs/{bucket}/gateway_jobs/{job_id}/inputs/assets/<rel_path>`.
3. **Selective HMMER Bypass + Hybrid NFS Template Lookup (`msa_pipeline_of3` & `predict_of3`)**:
   - In `msa_pipeline_of3`, any chain with pre-populated `main_msa_file_paths` and `template_alignment_file_path` skips `jackhmmer` (saving 10–20 minutes of CPU time), while any chain without local alignments (such as Chain `B` RNA below) still runs `nhmmer` against `/mnt/nfs/foldrun/rfam/rfam.fasta` and `/mnt/nfs/foldrun/rnacentral/rnacentral.fasta`!
   - Even when you supply a custom local `template_alignment_file_path` (e.g., `./templates/mcl1_homologs.sto`), `predict_of3` **still** resolves the referenced 3D template structures (e.g., `5fdr_A`) from `/mnt/nfs/foldrun/pdb_mmcif/mmcif_files/5fdr.cif` via `template_preprocessor_settings.structure_directory`.

In [ ]:
# @title 5.1 Bundle Local `.a3m` / `.sto` Alignments + Custom `runner.yaml`
from foldrun_shim.bundler import bundle_local_inputs

# Create local pre-computed MSA and Stockholm template alignment files for Protein Chain A
msas_dir = WORK_DIR / "msas"
tpl_dir = WORK_DIR / "templates"
msas_dir.mkdir(parents=True, exist_ok=True)
tpl_dir.mkdir(parents=True, exist_ok=True)

mcl1_seq = (
    "GDDELYRQSLEIISRYLREQATGAKDTKPMGRSGATSRKALETLRRVGDGVQRNHETAFQ"
    "GMLRKLDIKNEDDVKSLSRVMIHVFSDGVTNWGRIVTLISFGAFVAKHLKTINQESCIEP"
    "LAESITDVLVRTKRDWLVKQRGWDGFVEFFHVEDLEGG"
)

# 1. Local custom main MSA (.a3m) — first sequence must match the query protein sequence
(msas_dir / "mcl1_custom_uniref.a3m").write_text(
    f">rnp_atp_complex_A\n{mcl1_seq}\n>tr|Q07820|MCL1_HUMAN/171-327\n{mcl1_seq}\n",
    encoding="utf-8",
)

# 2. Local custom template alignment (.sto) in HMMER Stockholm format referencing PDB 5fdr_A
(tpl_dir / "mcl1_pdb_seqres.sto").write_text(
    f"# STOCKHOLM 1.0\n#=GS 5fdr_A/1-157 mol:protein\n5fdr_A/1-157 {mcl1_seq}\n//\n",
    encoding="utf-8",
)

# 3. Write query_precomputed.json:
#    - Chain A (protein) uses local bundled MSAs + template alignment
#    - Chain B (rna) has NO local MSA -> FoldRun automatically runs nhmmer against rfam & rnacentral on NFS!
#    - Chain L (ligand) uses CCD code ATP -> resolved via /mnt/nfs/foldrun/of3/ccd/components.bcif
precomputed_query = {
    "queries": {
        "rnp_atp_complex": {
            "chains": [
                {
                    "molecule_type": "protein",
                    "chain_ids": ["A"],
                    "sequence": mcl1_seq,
                    "main_msa_file_paths": ["./msas/mcl1_custom_uniref.a3m"],
                    "template_alignment_file_path": "./templates/mcl1_pdb_seqres.sto",
                },
                {
                    "molecule_type": "rna",
                    "chain_ids": ["B"],
                    "sequence": "GGGGCCACUAGGGACAGGAUGUUUUAGAGCUAGAAAUAGCAAGUUAAAAUAAGGCUAGUCCGUUAUCAACUUGAAAAAGUG",
                },
                {
                    "molecule_type": "ligand",
                    "chain_ids": ["L"],
                    "ccd_codes": ["ATP"],
                },
            ]
        }
    }
}
precomputed_json_path = WORK_DIR / "query_precomputed.json"
precomputed_json_path.write_text(json.dumps(precomputed_query, indent=2), encoding="utf-8")

# Inspect the bundle produced by foldrun-shim before upload:
bundled = bundle_local_inputs(
    model="openfold3",
    input_path=precomputed_json_path,
    cli_args=[
        "--num_diffusion_samples=5",
        "--num_model_seeds=2",
        "--use_templates=true",
    ],
    runner_yaml_path=runner_yaml_path,
)

print("Primary input filename       :", bundled.input_filename)
print("Bundled relative asset paths :", list(bundled.assets_map.keys()))
print("Compressed assets.tar.gz size:", len(bundled.assets_tar_bytes or b""), "bytes")
print("Parsed runner.yaml overrides :", bundled.runner_yaml_overrides)
print("Deterministic input_sha256   :", bundled.client_input_sha256)

if "example-uc.a.run.app" not in os.environ.get("FOLDRUN_GATEWAY_URL", ""):
    precomputed_job = openfold3.predict(
        precomputed_json_path,
        output_dir="./of3_precomputed_out",
        runner_yaml="runner.yaml",
        num_diffusion_samples=5,
        num_model_seeds=2,
        use_templates=True,
    )
    display(precomputed_job)

## 6. Inspecting Materialized Native OpenFold 3 Outputs & Confidence Metrics

When a FoldRun OpenFold 3 job succeeds, `_package_openfold3_outputs()` in [`foldrun_app/gateway/output_packager.py`](../../../foldrun-agent/foldrun_app/gateway/output_packager.py) streams all per-seed outputs from GCS into `native_outputs.tar.gz`, preserving OpenFold 3's exact native output directory layout under `--output_dir`:

```text
./of3_out/
├── .foldrun_job.json
└── rnp_atp_complex/                                                     # <query_name>
    ├── seed_42/                                                         # seed_<seed>
    │   ├── rnp_atp_complex_seed_42_sample_1_model.cif                   # 3D structure (mmCIF with per-atom B-factor pLDDT)
    │   ├── rnp_atp_complex_seed_42_sample_1_confidences_aggregated.json # Whole-complex ranking & confidence metrics
    │   ├── rnp_atp_complex_seed_42_sample_1_confidences.json            # Full per-atom pLDDT, PAE, and PDE arrays
    │   ├── ...
    │   └── rnp_atp_complex_seed_42_sample_5_model.cif
    └── seed_100/
        ├── rnp_atp_complex_seed_100_sample_1_model.cif
        ├── rnp_atp_complex_seed_100_sample_1_confidences_aggregated.json
        └── rnp_atp_complex_seed_100_sample_1_confidences.json
```

### OpenFold 3 Confidence & Sample Ranking Metrics (`*_confidences_aggregated.json`)
- **`sample_ranking_score`**: Composite AlphaFold 3 / OpenFold 3 ranking score ($0.8 \times \text{ipTM} + 0.2 \times \text{pTM} - 100 \times \text{has\_clash}$). Note: for single-chain monomers where $\text{ipTM} \approx 0$, FoldRun's `predict_of3` component automatically ranks samples by `ptm` instead so monomer ranking reflects fold quality.
- **`avg_plddt`**: Mean per-atom Predicted Local Distance Difference Test ($0\text{–}100$).
- **`ptm`** & **`iptm`**: Predicted TM-score for the full complex and across inter-chain interfaces ($0\text{–}1$).
- **`gpde`**: Global Predicted Distance Error (Å, lower is better).
- **`chain_ptm`**, **`chain_pair_iptm`**, **`bespoke_iptm`**: Per-chain and pairwise interface confidence breakdowns.

In [ ]:
# @title 6.1 Parse `<query_name>/seed_<seed>/` Outputs & Rank Predictions Across Seeds
import math
import os

import numpy as np

query_name = "rnp_atp_complex"
seeds = [42, 100]
num_samples = 5

existing_aggs = sorted((output_dir / query_name).glob("seed_*/*_confidences_aggregated.json"))
if not existing_aggs:
    any_aggs = sorted(output_dir.glob("*/seed_*/*_confidences_aggregated.json"))
    if any_aggs:
        query_name = any_aggs[0].parent.parent.name
        existing_aggs = sorted(
            (output_dir / query_name).glob("seed_*/*_confidences_aggregated.json")
        )

synthetic_demo_mode = os.environ.get("FOLDRUN_SYNTHETIC_DEMO", "").strip().lower() == "true"

if not existing_aggs:
    materialized = (
        sorted(
            p.relative_to(output_dir).as_posix()
            for p in output_dir.rglob("*")
            if p.is_file() and p.name != ".foldrun_job.json"
        )
        if output_dir.exists()
        else []
    )
    if not synthetic_demo_mode:
        raise FileNotFoundError(
            f"No OpenFold 3 prediction outputs (seed_*/*_confidences_aggregated.json) found in {output_dir.resolve()}.\n"
            f"Files currently in {output_dir}: {materialized or ['<empty>']}\n"
            f"Run the OpenFold 3 prediction cell above against a live FoldRun Gateway first "
            f"(or set FOLDRUN_SYNTHETIC_DEMO=true strictly to test notebook chart layout offline)."
        )

    print("=" * 80)
    print("SYNTHETIC DEMO DATA (NOT AN OPENFOLD 3 PREDICTION) — FOLDRUN_SYNTHETIC_DEMO=true")
    print("=" * 80)
    rng = np.random.default_rng(42)
    n_prot, n_rna, n_dna, n_lig = 36, 20, 12, 6
    n_total = n_prot + n_rna + n_dna + n_lig

    for s_idx, seed_val in enumerate(seeds):
        seed_dir = output_dir / query_name / f"seed_{seed_val}"
        seed_dir.mkdir(parents=True, exist_ok=True)
        for sample_idx in range(1, num_samples + 1):
            prefix = f"{query_name}_seed_{seed_val}_sample_{sample_idx}"
            quality_shift = 0.03 * (num_samples - sample_idx) - 0.015 * s_idx

            plddt_prot = np.clip(
                88.0
                + 6.0 * np.sin(np.linspace(0, 3.14, n_prot))
                + rng.normal(0, 2.0, n_prot)
                + quality_shift * 40,
                50,
                98,
            )
            plddt_rna = np.clip(
                81.0
                + 5.0 * np.cos(np.linspace(0, 3.14, n_rna))
                + rng.normal(0, 2.5, n_rna)
                + quality_shift * 40,
                45,
                95,
            )
            plddt_dna = np.clip(76.0 + rng.normal(0, 3.0, n_dna) + quality_shift * 40, 45, 92)
            plddt_lig = np.clip(85.0 + rng.normal(0, 2.0, n_lig) + quality_shift * 40, 55, 96)
            plddt_all = np.concatenate([plddt_prot, plddt_rna, plddt_dna, plddt_lig]).round(2)

            pae_matrix = np.full((n_total, n_total), 11.5)
            pae_matrix[:n_prot, :n_prot] = 3.2
            pae_matrix[n_prot : n_prot + n_rna, n_prot : n_prot + n_rna] = 4.1
            pae_matrix[:n_prot, n_prot : n_prot + n_rna] = 5.4
            pae_matrix[n_prot : n_prot + n_rna, :n_prot] = 5.4
            pae_matrix[n_prot + n_rna :, n_prot + n_rna :] = 4.8
            pae_matrix = np.clip(
                pae_matrix + rng.uniform(-0.8, 0.8, (n_total, n_total)) - quality_shift * 15,
                0.8,
                30.0,
            )
            np.fill_diagonal(pae_matrix, 0.5)
            pde_matrix = np.clip(pae_matrix * 0.45, 0.3, 15.0)

            conf_full = {
                "plddt": plddt_all.tolist(),
                "pae": np.round(pae_matrix, 2).tolist(),
                "pde": np.round(pde_matrix, 2).tolist(),
                "chain_labels": (["A"] * n_prot)
                + (["B"] * n_rna)
                + (["C"] * n_dna)
                + (["L"] * 4)
                + (["M"] * 2),
            }
            (seed_dir / f"{prefix}_confidences.json").write_text(
                json.dumps(conf_full), encoding="utf-8"
            )

            ptm = round(float(0.84 + quality_shift), 4)
            iptm = round(float(0.81 + quality_shift * 1.1), 4)
            ranking_score = round(0.8 * iptm + 0.2 * ptm, 4)
            conf_agg = {
                "avg_plddt": round(float(np.mean(plddt_all)), 2),
                "gpde": round(float(np.mean(pde_matrix)), 3),
                "ptm": ptm,
                "iptm": iptm,
                "disorder": 0.04,
                "has_clash": 0.0,
                "sample_ranking_score": ranking_score,
                "chain_ptm": {
                    "A": round(ptm + 0.04, 3),
                    "B": round(ptm - 0.02, 3),
                    "C": round(ptm - 0.05, 3),
                },
                "chain_pair_iptm": {"A_B": iptm, "A_L": round(iptm + 0.03, 3)},
                "bespoke_iptm": {"A": iptm, "B": iptm},
            }
            (seed_dir / f"{prefix}_confidences_aggregated.json").write_text(
                json.dumps(conf_agg, indent=2), encoding="utf-8"
            )

            cif_lines = [
                f"data_{prefix}",
                "#",
                "loop_",
                "_atom_site.group_PDB",
                "_atom_site.id",
                "_atom_site.type_symbol",
                "_atom_site.label_atom_id",
                "_atom_site.label_alt_id",
                "_atom_site.label_comp_id",
                "_atom_site.label_asym_id",
                "_atom_site.label_entity_id",
                "_atom_site.label_seq_id",
                "_atom_site.pdbx_PDB_ins_code",
                "_atom_site.Cartn_x",
                "_atom_site.Cartn_y",
                "_atom_site.Cartn_z",
                "_atom_site.occupancy",
                "_atom_site.B_iso_or_equiv",
                "_atom_site.auth_seq_id",
                "_atom_site.auth_comp_id",
                "_atom_site.auth_asym_id",
                "_atom_site.pdbx_PDB_model_num",
            ]
            atom_id = 1
            for i in range(n_prot):
                angle = i * (2.0 * math.pi / 3.6)
                x = round(6.0 * math.cos(angle), 3)
                y = round(6.0 * math.sin(angle), 3)
                z = round(i * 1.5, 3)
                b_fac = round(float(plddt_prot[i]), 2)
                cif_lines.append(
                    f"ATOM {atom_id} C CA . ALA A 1 {i + 1} ? {x} {y} {z} 1.00 {b_fac} {i + 1} ALA A 1"
                )
                atom_id += 1
            for j in range(n_rna):
                angle = j * (2.0 * math.pi / 10.0)
                x = round(14.0 + 5.5 * math.cos(angle), 3)
                y = round(5.5 * math.sin(angle), 3)
                z = round(j * 2.4, 3)
                b_fac = round(float(plddt_rna[j]), 2)
                cif_lines.append(
                    f"ATOM {atom_id} P P . G B 2 {j + 1} ? {x} {y} {z} 1.00 {b_fac} {j + 1} G B 1"
                )
                atom_id += 1
            atp_coords = [
                ("P", "PA", 9.5, 2.0, 18.0),
                ("P", "PB", 10.5, 3.2, 19.2),
                ("P", "PG", 11.2, 1.8, 20.4),
                ("O", "O5'", 8.4, 1.2, 17.1),
                ("C", "C5'", 7.3, 1.8, 16.4),
                ("N", "N9", 6.5, 0.9, 15.4),
            ]
            for k, (elem, aname, lx, ly, lz) in enumerate(atp_coords):
                b_fac = round(float(plddt_lig[k]), 2)
                cif_lines.append(
                    f"HETATM {atom_id} {elem} {aname} . ATP L 3 . ? {lx} {ly} {lz} 1.00 {b_fac} 1 ATP L 1"
                )
                atom_id += 1
            cif_lines.append("#")
            (seed_dir / f"{prefix}_model.cif").write_text(
                "\n".join(cif_lines) + "\n", encoding="utf-8"
            )

# Discover and rank all materialized predictions across seeds
records = []
discovered_seeds = set()
for agg_path in sorted((output_dir / query_name).glob("seed_*/*_confidences_aggregated.json")):
    agg = json.loads(agg_path.read_text(encoding="utf-8"))
    stem = agg_path.name.replace("_confidences_aggregated.json", "")
    cif_path = agg_path.with_name(f"{stem}_model.cif")
    conf_path = agg_path.with_name(f"{stem}_confidences.json")
    discovered_seeds.add(agg_path.parent.name)
    records.append(
        {
            "sample_id": stem,
            "seed": agg_path.parent.name,
            "sample_ranking_score": agg.get("sample_ranking_score", 0.0),
            "ptm": agg.get("ptm", 0.0),
            "iptm": agg.get("iptm", 0.0) if agg.get("iptm") is not None else agg.get("ptm", 0.0),
            "avg_plddt": agg.get("avg_plddt", 0.0),
            "gpde": agg.get("gpde", 0.0),
            "has_clash": agg.get("has_clash", 0.0),
            "cif_path": cif_path,
            "conf_path": conf_path,
            "agg_path": agg_path,
        }
    )

records.sort(key=lambda r: r["sample_ranking_score"], reverse=True)
print(
    f"Discovered {len(records)} OpenFold 3 predictions across {len(discovered_seeds)} seeds in {output_dir / query_name}:\n"
)
print(
    f"{'Rank':<5} {'Sample ID':<42} {'Ranking Score':<15} {'ipTM':<8} {'pTM':<8} {'avg_pLDDT':<10} {'gPDE (Å)':<9}"
)
print("-" * 102)
for idx, r in enumerate(records, 1):
    print(
        f"{idx:<5} {r['sample_id']:<42} {r['sample_ranking_score']:<15.4f} "
        f"{r['iptm']:<8.4f} {r['ptm']:<8.4f} {r['avg_plddt']:<10.2f} {r['gpde']:<9.3f}"
    )

best_record = records[0]
print(f"\nTop-Ranked Structure : {best_record['cif_path']}")

In [ ]:
# @title 6.2 Plot OpenFold 3 Confidence Diagnostics (pLDDT, 2D PAE/PDE Matrix & Multi-Seed Ranking)
best_conf = json.loads(best_record["conf_path"].read_text(encoding="utf-8"))
plddt_vals = np.array(best_conf["plddt"])
err_matrix_key = "pae" if "pae" in best_conf else "pde"
err_matrix_label = "PAE" if err_matrix_key == "pae" else "PDE"
pae_vals = np.array(best_conf[err_matrix_key])

try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    if os.environ.get("FOLDRUN_SYNTHETIC_DEMO", "").strip().lower() == "true":
        fig.suptitle(
            "SYNTHETIC DEMO DATA (NOT AN OPENFOLD 3 PREDICTION)",
            color="#b31412",
            fontweight="bold",
        )

    # Panel 1: Per-Token pLDDT Profile across Protein (A), RNA (B), DNA (C), and Ligands (L, M)
    ax0 = axes[0]
    x_idx = np.arange(1, len(plddt_vals) + 1)
    ax0.plot(x_idx, plddt_vals, color="#1a73e8", linewidth=2, label="pLDDT")
    ax0.axhspan(90, 100, color="#0053d6", alpha=0.12, label="Very High (>90)")
    ax0.axhspan(70, 90, color="#65cbf3", alpha=0.12, label="Confident (70-90)")
    ax0.axhspan(50, 70, color="#ffdb13", alpha=0.12, label="Low (50-70)")
    ax0.axvline(36.5, color="#5f6368", linestyle="--", linewidth=1)
    ax0.axvline(56.5, color="#5f6368", linestyle="--", linewidth=1)
    ax0.axvline(68.5, color="#5f6368", linestyle="--", linewidth=1)
    ax0.set_ylim(40, 100)
    ax0.set_xlabel("Token Index (Protein A | RNA B | DNA C | Ligands)")
    ax0.set_ylabel("pLDDT (0-100)")
    ax0.set_title(f"Top Sample pLDDT (mean = {best_record['avg_plddt']:.1f})")
    ax0.legend(loc="lower left", fontsize=8)

    # Panel 2: 2D Predicted Aligned/Distance Error (PAE/PDE) Matrix
    ax1 = axes[1]
    vmax_val = 30 if err_matrix_key == "pae" else 15
    im = ax1.imshow(pae_vals, cmap="Greens_r", vmin=0, vmax=vmax_val, origin="upper")
    ax1.axhline(35.5, color="white", linestyle="--", linewidth=1)
    ax1.axvline(35.5, color="white", linestyle="--", linewidth=1)
    ax1.set_xlabel("Scored Token")
    ax1.set_ylabel("Aligned Token")
    ax1.set_title(f"Predicted {err_matrix_label} Matrix (ipTM = {best_record['iptm']:.3f})")
    fig.colorbar(im, ax=ax1, fraction=0.046, pad=0.04, label=f"Expected {err_matrix_label} (Å)")

    # Panel 3: Multi-Seed Sample Ranking Comparison
    ax2 = axes[2]
    top_k = records[:10]
    unique_seeds = sorted({r["seed"] for r in top_k})
    first_seed = unique_seeds[0] if unique_seeds else "seed_42"
    labels = [r["sample_id"].replace(f"{query_name}_", "") for r in top_k][::-1]
    scores = [r["sample_ranking_score"] for r in top_k][::-1]
    colors = ["#1e8e3e" if first_seed in lbl else "#1a73e8" for lbl in labels]
    ax2.barh(labels, scores, color=colors)
    ax2.set_xlim(min(scores) - 0.03, max(scores) + 0.02)
    ax2.set_xlabel("OpenFold 3 sample_ranking_score (0.8*ipTM + 0.2*pTM)")
    ax2.set_title(f"Parallel Multi-Seed Ranking ({', '.join(unique_seeds)})")

    plt.tight_layout()
    plt.show()
except ImportError:
    print(
        f"Loaded confidence arrays: pLDDT shape={plddt_vals.shape}, {err_matrix_label} shape={pae_vals.shape}"
    )

In [ ]:
# @title 6.3 Interactive 3D Visualization of Top-Ranked `.cif` Structure (`py3Dmol`)
cif_text = best_record["cif_path"].read_text(encoding="utf-8")

try:
    import py3Dmol

    view = py3Dmol.view(width=860, height=480)
    view.addModel(cif_text, "cif")

    # Protein Chain A: Cartoon colored by pLDDT (stored in mmCIF B-factor column)
    view.setStyle(
        {"chain": "A"},
        {
            "cartoon": {
                "colorscheme": {
                    "prop": "b",
                    "gradient": "roygb",
                    "min": 50,
                    "max": 95,
                }
            }
        },
    )
    # RNA Chain B: Cartoon + nucleotide sticks in teal
    view.setStyle(
        {"chain": "B"},
        {
            "cartoon": {"color": "#00897b"},
            "stick": {"radius": 0.18, "colorscheme": "tealCarbon"},
        },
    )
    # ATP Ligand Chain L: Ball-and-stick in magenta
    view.setStyle(
        {"chain": "L"},
        {
            "stick": {"radius": 0.28, "colorscheme": "magentaCarbon"},
            "sphere": {"scale": 0.3},
        },
    )
    view.zoomTo()
    view.show()
except ImportError:
    print("Install py3Dmol (`%pip install py3Dmol`) to render the interactive 3D structure viewer.")
    print("\nFirst 15 lines of materialized top-ranked mmCIF structure:")
    print("\n".join(cif_text.splitlines()[:15]))

## 7. Summary: Where FoldRun Provides an Advantage Over Public OpenFold 3 Setups

| Advantage Area | Public OpenFold 3 Notebooks / Local CLI | FoldRun (`foldrun-shim` + `foldrun-gateway`) |
| :--- | :--- | :--- |
| **1. Zero Local Setup of 9 Databases (Protein + RNA + PDB mmCIF + CCD)** | Full Protein + RNA + Template folding requires downloading >2.5 TB of databases (`uniref90`, `uniprot`, `mgnify`, `pdb_mmcif`, `pdb_seqres`, `rfam`, `rnacentral`, `components.bcif`, `of3-p2-155k.pt`) and compiling `hmmer`/`hhsuite`/CUDA kernels. Public ColabFold MSA mode (`--use-msa-server`) **only aligns protein chains** (no RNA MSAs). | Pre-mounts all **9 databases** on high-IOPS `BASIC_SSD` Cloud Filestore NFS (`/mnt/nfs/foldrun`). Runs both protein (`jackhmmer`) and RNA (`nhmmer` against `rfam` & `rnacentral`) alignments privately inside your GCP VPC, with per-chain SHA-256 caching in `/mnt/nfs/foldrun/of3_msas_cache/`. |
| **2. Safe `runner.yaml` Merging & Automatic Template Resolution** | Requires manually configuring local template structure directories in `runner.yaml` or fetching `.cif` files over the internet from RCSB during inference. | Validates scientific `runner.yaml` overrides (`model_update`, `low_mem`, `plddt`, `pae`, `experiment_settings`) via `flag_allowlist.py` and automatically wires `template_preprocessor_settings.structure_directory = /mnt/nfs/foldrun/pdb_mmcif/mmcif_files` (`fetch_missing_structures: false`). |
| **3. Parallel Multi-Seed A100 40GB / 80GB Execution** | Runs multiple seeds sequentially on a single local GPU or requires manual SLURM/multi-GPU cluster configuration. | Fans out `--num_model_seeds` in parallel across dedicated **NVIDIA A100 40GB (`a2-highgpu-1g`)** or **A100 80GB (`a2-ultragpu-1g`)** GPUs (auto-selected by token count `count_tokens()`) using Vertex AI `dsl.ParallelFor` and Dynamic Workload Scheduler (`FLEX_START`), patching `query_data["seeds"] = [seed_value]` per worker. |
| **4. Interrupt Resilience (`.foldrun_job.json`) & Job Deduplication** | If a Colab session times out or laptop Wi-Fi drops during a 45-minute folding run, progress is lost and re-running starts from scratch. | Persists `<output_dir>/.foldrun_job.json` with `job_id` and canonical `input_sha256` upon `202 Accepted`. Re-running the cell automatically re-attaches to the running or completed Vertex AI `PipelineJob` and streams the native output archive into `--output_dir`. |